# Spike sorting
&#x26a0;&#xfe0f; **Note** &#x26a0;&#xfe0f;

This notebook demonstrates a spike sorting workflow with kilosort4.

- Spike sorting is computationally intensive and typically benefits from a GPU. Currently, spikeinterface supports NVIDIA GPUs.
- For GPU, please install the cuda environment, `conda env create -f environment_cuda.yml`
- Running this pipeline on CPU is possible but may be slow.
- The dateset used for this notebook is ~2GB.

**Recording details**
- Probe: 64-channel silicon probe ASSY-236-H5 layout (Cambridge Neurotech)
- Location: Primary Motor Cortex
- Amplifier: mini-amp-64 (Cambridge Neurotech)
- Acquisition system: Open Ephys acquisition board

## CUDA availability
If you've created the `neurokinematics_cuda` environment from `environment_cuda.yml`, first check that CUDA is available. You can do this step if using the plain `neurokinematics` environment as well. If unavailable, kilosort will run on the CPU (much slower).

In [ ]:
import torch

# check cuda availability
if torch.cuda.is_available():
    ngpus = torch.cuda.device_count()
    print(f"CUDA available: {ngpus} GPU(s)")
    
    for i in range(ngpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

else:
    print("WARNING: CUDA unavailable. Spike sorting will run on CPU.")



## Download open ephys data from huggingface hub
Data for this demo is available publically on huggingface hub. Data will by default be stored in the current working directory. To change this, amend th `root_path` variable to your desired location.

Again, this dataset will be roughly ~2GB, so downloading may take time.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

# create directory if
data_dir = Path.cwd() / 'sample_oephys_data'

hugging_face_repo = 'cjblack1111/sample_oephys_data'
repo_type = 'dataset'

if not any(data_dir.glob('*')):
    # download entire repo
    print(f"Downloading dataset (~3GB) to {data_dir.resolve()}, this may take a while...")
    oephys_path = snapshot_download(
        repo_id = "cjblack1111/sample_oephys_data",
        repo_type = repo_type,
        local_dir = data_dir,
        local_dir_use_symlinks = False
    )

## Run spike sorting
Spike sorting is performed using the `sort` function from the `sorting` subpackage. This function uses the `run_sorter` function from `spikeinterface`. It requires a spike sorting config file, and the path pointing toward the ephys data used for sorting.
- The spike config will need to be stored in the `configs/spk_sorting_cfg` folder in this project's main directory.
- At the moment, spike sorting has been tested with kilosort4 on data aquired with Open Ephys. In this instance, the `data_path` variable will point to the relevant folder containing a `.oebin` file.

In [ ]:
from neurokinematics.ephys.spikes.sorting import sort

# use default spike sorting cfg
cfg_file = 'demo_spike_sorting_cfg.yaml'
data_path = data_dir / 'Record Node 109' / 'experiment1' / 'recording1' # set data path toward .oebin - this will depend on which record node your spike filtered data was recorded on

# sort spikes
sorter, recording, probe, analyzer = sort(data_path=data_path, cfg_file=cfg_file)

## Plotting results
This pacakge also utilises the `spikeinterface.widget` module to help handle plotting with sorting and analyzer objects. These functions exist in the `plotting` module of the `spikes` subpackage:

In [ ]:
from neurokinematics.ephys.spikes.plotting import plot_autocorrelogram, plot_waveforms

We can check the unit ids, and then plot select waveforms and autocorrelograms:

In [ ]:
print(f"Unit ids: {sorter.unit_ids}") # check unit ids
plot_waveforms(analyzer, unit_ids = [4, 16, 18]) # plot waveforms of select units along probe positions
plot_autocorrelogram(sorter, unit_ids = [4, 16, 18]) # plot corresponding autocorrelograms